### Experiment V1: Frozen ResNet50 Baseline

#### Objective

Build a baseline brain tumor classification model using transfer learning.

Model:
- ResNet50 pretrained on ImageNet
- Frozen feature extractor
- GlobalAveragePooling2D
- Dense(256, ReLU)
- Dropout(0.3)
- Dense(4, Softmax)

Goal:
- Establish baseline performance
- Identify weaknesses before fine-tuning

#### Section 2: Dataset Information
##### Dataset Summary

Classes:
- glioma
- meningioma
- no_tumor
- pituitary

Class Distribution

glioma      : 1147
meningioma  : 1329
no_tumor    : 1067
pituitary   : 1457

Observation:
Dataset is reasonably balanced.
No class weighting used in baseline experiment.

#### Section 3: Dataset Loading

In [ ]:
from pathlib import Path
import tensorflow as tf

DATASET_DIR = Path(
    "../artifacts/data_ingestion/brisc2025/brisc2025/classification_task"
)

train_dir = DATASET_DIR / "train"
test_dir = DATASET_DIR / "test"

##### TRAIN/TEST/VALIDATION SPLIT
##### Data Pipeline

- image_dataset_from_directory()
- image_size=(224,224)
- batch_size=32
- validation_split=0.2
- shuffle=True for training
- shuffle=False for testing


In [4]:


print(tf.__version__)

2.21.0


In [3]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(111)

import warnings
warnings.filterwarnings('ignore')

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

Found 5000 files belonging to 4 classes.
Using 4000 files for training.


In [6]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

Found 5000 files belonging to 4 classes.
Using 1000 files for validation.


In [7]:
test_ds =tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size = 32,
    shuffle = False
)

Found 1000 files belonging to 4 classes.


In [8]:
print(train_ds.class_names)
for images, labels in train_ds.take(1):
    print("Images Shape :", images.shape)
    print("Labels Shape :", labels.shape)
    print("Images Dtype :", images.dtype)
    print("Labels Dtype :", labels.dtype)

['glioma', 'meningioma', 'no_tumor', 'pituitary']
Images Shape : (32, 224, 224, 3)
Labels Shape : (32,)
Images Dtype : <dtype: 'float32'>
Labels Dtype : <dtype: 'int32'>


In [9]:
for image, label in test_ds.take(1):
  print(image.shape)
  print(label.shape)

(32, 224, 224, 3)
(32,)


#### Section 4: Model Architecture

In [10]:
base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    tf.keras.layers.Lambda(
        tf.keras.applications.resnet50.preprocess_input
    ),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(
        256,
        activation="relu"
    ),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(
        4,
        activation="softmax"
    )
])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 94s 1us/step



In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,113,284 (91.98 MB)

 Trainable params: 525,572 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

##### Architecture Rationale

Why ResNet50?

- Pretrained on ImageNet
- Strong feature extractor
- Works well with small datasets

Why GlobalAveragePooling2D?

- Reduces parameters
- Reduces overfitting
- Better generalization

Why Dropout(0.3)?

- Helps prevent overfitting

#### Section 5: Training Configuration
##### Training Configuration

Optimizer:
Adam

Loss:
SparseCategoricalCrossentropy

Metrics:
Accuracy

Callbacks:
EarlyStopping

Parameters:
Epochs = 20
Batch Size = 32
Patience = 5

In [12]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [13]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 426s 3s/step - accuracy: 0.8033 - loss: 0.5442 - val_accuracy: 0.8770 - val_loss: 0.2914
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 464s 4s/step - accuracy: 0.8878 - loss: 0.2815 - val_accuracy: 0.9210 - val_loss: 0.2145
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 389s 3s/step - accuracy: 0.9153 - loss: 0.2215 - val_accuracy: 0.9190 - val_loss: 0.2183
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 373s 3s/step - accuracy: 0.9273 - loss: 0.1920 - val_accuracy: 0.9380 - val_loss: 0.1712
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 341s 3s/step - accuracy: 0.9427 - loss: 0.1533 - val_accuracy: 0.9410 - val_loss: 0.1476
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 361s 3s/step - accuracy: 0.9450 - loss: 0.1391 - val_accuracy: 0.8960 - val_loss: 0.2637
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 328s 3s/step - accuracy: 0.9532 - loss: 0.1192 - val_accuracy: 0.9450 - val_loss: 0.1481
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 359s 3s/step - accuracy: 0.9640 - loss: 0.0984 - val_accu

##### Section 6: Training Results
- Best Validation Accuracy = 95.6%
- Best Validation Loss = 0.1345
- Stopped at Epoch 13

#### Section 7: Test Evaluation

In [15]:
test_loss, test_acc = model.evaluate(test_ds)
print(test_acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.9240 - loss: 0.2286
0.9240000247955322


Test Accuracy = 92.4%

In [17]:
import numpy as np
predictions = model.predict(test_ds)

y_pred = np.argmax(
    predictions,
    axis=1
)

32/32 ━━━━━━━━━━━━━━━━━━━━ 96s 3s/step


In [18]:
y_true = np.concatenate(
    [labels.numpy() for _, labels in test_ds]
)

In [19]:
print(y_true.shape)
print(y_pred.shape)

(1000,)
(1000,)


#### Section 8: Classification Report

In [21]:

from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=test_ds.class_names
    )
)

              precision    recall  f1-score   support

      glioma       0.97      0.80      0.87       254
  meningioma       0.84      0.95      0.89       306
    no_tumor       0.97      1.00      0.99       140
   pituitary       0.96      0.97      0.97       300

    accuracy                           0.92      1000
   macro avg       0.94      0.93      0.93      1000
weighted avg       0.93      0.92      0.92      1000



#### Section 9: Confusion Matrix

In [22]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)

[[202  48   0   4]
 [  5 290   4   7]
 [  0   0 140   0]
 [  1   7   0 292]]


##### Key Findings

1. no_tumor achieved perfect recall (100%).

2. pituitary achieved excellent recall (97%).

3. Glioma recall is only 80%.

4. Largest confusion:
   glioma → meningioma (48 cases)

Interpretation

The model struggles to distinguish
glioma and meningioma MRI images.

This is the dominant source of error.

#####  Engineering Decision

Current Model Status

Test Accuracy: 92.4%
Macro F1: 93%

Decision

Do not modify architecture.

Proceed with fine-tuning ResNet50.

Plan

- Unfreeze last 30 layers
- Learning rate = 1e-5
- Train for 10 epochs
- Compare against baseline

Primary Objective

Increase glioma recall while maintaining overall accuracy.